# 🌬️ UK Wind Power Reliability Analysis

**Question**: Based on historical actual wind generation data, how many MW of wind power can we **reliably** expect to be available to meet electricity demand?

**Approach**: Analyse the empirical distribution of actual wind generation to identify a "firm capacity" — a level that can be counted on with high probability. This concept is related to the Capacity Credit or Firm Capacity of wind power in power system planning.

**Dataset**: Actual UK national wind generation, January 2024 (FUELHH, 30-min resolution)

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats as scipy_stats
import warnings

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1a0f',
    'axes.facecolor': '#111a11',
    'axes.edgecolor': '#2a3a2a',
    'axes.labelcolor': '#8dab82',
    'xtick.color': '#4d6647',
    'ytick.color': '#4d6647',
    'grid.color': '#1e2e1e',
    'grid.linestyle': '--',
    'text.color': '#e8f5e2',
    'font.size': 11,
    'axes.titlecolor': '#e8f5e2',
    'legend.facecolor': '#111a11',
    'legend.edgecolor': '#2a3a2a',
})

WIND_COLOR = '#4ade80'
ACTUAL_COLOR = '#60a5fa'
WARN_COLOR = '#fbbf24'
NEG_COLOR = '#f87171'

print('Setup complete.')

## 1. Fetch & Prepare Actual Generation Data

In [ ]:
BASE_URL = 'https://data.elexon.co.uk/bmrs/api/v1'

def fetch_stream(endpoint: str, params: dict) -> list:
    url = f'{BASE_URL}/{endpoint}'
    resp = requests.get(url, params=params, timeout=90)
    resp.raise_for_status()
    data = resp.json()
    if isinstance(data, list):
        return data
    return data.get('data', data.get('items', []))

print('Fetching actual generation...')
actual_raw = fetch_stream('datasets/FUELHH/stream', {
    'publishDateTimeFrom': '2024-01-01T00:00Z',
    'publishDateTimeTo': '2024-01-31T23:59Z',
})

df_raw = pd.DataFrame(actual_raw)

# Keep WIND only
df = df_raw[df_raw['fuelType'].str.upper() == 'WIND'].copy()
df['startTime'] = pd.to_datetime(df['startTime'], utc=True)
df['generation'] = pd.to_numeric(df['generation'], errors='coerce')
df = df[['startTime', 'generation']].drop_duplicates('startTime').sort_values('startTime').reset_index(drop=True)

print(f'Records: {len(df)}')
print(f'Date range: {df["startTime"].min()} → {df["startTime"].max()}')
print(f'Generation: {df["generation"].min():.0f} – {df["generation"].max():.0f} MW (avg: {df["generation"].mean():.0f} MW)')
df.head()

## 2. Generation Time Series Overview

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(df['startTime'], df['generation'], alpha=0.2, color=WIND_COLOR)
ax.plot(df['startTime'], df['generation'], color=WIND_COLOR, linewidth=1.2, label='Wind Generation')

# Annotate percentiles
for p, c, ls in [(10, NEG_COLOR, '--'), (50, WARN_COLOR, '--'), (90, ACTUAL_COLOR, '--')]:
    val = df['generation'].quantile(p/100)
    ax.axhline(val, color=c, linestyle=ls, linewidth=1.5, alpha=0.8, label=f'P{p}: {val:.0f} MW')

ax.set_xlabel('Date (January 2024)')
ax.set_ylabel('Wind Power Generation (MW)')
ax.set_title('UK National Wind Power — Actual Generation (January 2024)')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('wind_timeseries.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: wind_timeseries.png')

## 3. Distribution Analysis

In [ ]:
gen = df['generation'].dropna()

# Percentile table
percentiles = [1, 5, 10, 25, 50, 75, 90, 95, 99]
pct_values = {p: gen.quantile(p / 100) for p in percentiles}

print('Percentile Table (Wind Generation, January 2024):')
print(f'{"Percentile":>12} | {"MW":>10} | Notes')
print('-' * 50)
for p, v in pct_values.items():
    note = ''
    if p == 5: note = ' ← Conservative firm capacity'
    elif p == 10: note = ' ← Moderate firm capacity'
    elif p == 50: note = ' ← Median (50% of time above this)'
    print(f'  P{p:2d}         | {v:10.0f} | {note}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Wind Generation Distribution — January 2024', fontsize=13)

# Histogram + KDE
ax1 = axes[0]
ax1.hist(gen, bins=40, color=WIND_COLOR, alpha=0.7, density=True, label='Observed')
kde_x = np.linspace(gen.min(), gen.max(), 300)
kde = scipy_stats.gaussian_kde(gen)
ax1.plot(kde_x, kde(kde_x), color=ACTUAL_COLOR, linewidth=2.5, label='KDE')
for p, c in [(10, NEG_COLOR), (50, WARN_COLOR)]:
    val = gen.quantile(p/100)
    ax1.axvline(val, color=c, linestyle='--', linewidth=2, label=f'P{p}: {val:.0f} MW')
ax1.set_xlabel('Generation (MW)')
ax1.set_ylabel('Density')
ax1.set_title('Generation Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# CDF (exceedance curve)
ax2 = axes[1]
sorted_gen = np.sort(gen)
exceedance = (1 - np.arange(1, len(sorted_gen)+1) / len(sorted_gen)) * 100
ax2.plot(sorted_gen, exceedance, color=WIND_COLOR, linewidth=2.5, label='Exceedance')
for p, c, label in [(10, NEG_COLOR, '90% exceedance (P10)'),
                     (50, WARN_COLOR, '50% exceedance (P50)')]:
    val = gen.quantile(p/100)
    ax2.axvline(val, color=c, linestyle='--', linewidth=2, label=f'{label}\n{val:.0f} MW')
ax2.set_xlabel('Generation (MW)')
ax2.set_ylabel('% of time generation exceeds x')
ax2.set_title('Exceedance Probability (Duration Curve)')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter())
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('generation_distribution.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: generation_distribution.png')

## 4. Load Duration Curve

In [ ]:
sorted_desc = np.sort(gen)[::-1]
hours = np.linspace(0, len(sorted_desc) * 0.5, len(sorted_desc))  # 30-min intervals → hours

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(hours, sorted_desc, alpha=0.2, color=WIND_COLOR)
ax.plot(hours, sorted_desc, color=WIND_COLOR, linewidth=2.5, label='Wind Generation Duration Curve')

# Mark key levels
total_hours = hours[-1]
for pct, c, ls in [(90, NEG_COLOR, '--'), (75, WARN_COLOR, '--'), (50, ACTUAL_COLOR, '--')]:
    val = gen.quantile(1 - pct/100)
    ax.axhline(val, color=c, linestyle=ls, linewidth=1.5,
               label=f'Available {pct}% of hours: {val:.0f} MW')

ax.set_xlabel('Hours in month (sorted descending)')
ax.set_ylabel('Wind Generation (MW)')
ax.set_title('Wind Power Load Duration Curve — January 2024')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('load_duration_curve.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: load_duration_curve.png')

## 5. Variability by Time of Day and Day of Week

In [ ]:
df['hour'] = df['startTime'].dt.hour
df['dow'] = df['startTime'].dt.day_name()
df['date'] = df['startTime'].dt.date

hourly = df.groupby('hour')['generation'].agg(['mean', 'median',
    ('p10', lambda x: x.quantile(0.10)),
    ('p25', lambda x: x.quantile(0.25)),
    ('p75', lambda x: x.quantile(0.75)),
    ('p90', lambda x: x.quantile(0.90)),
]).reset_index()

fig, ax = plt.subplots(figsize=(14, 6))
ax.fill_between(hourly['hour'], hourly['p10'], hourly['p90'], alpha=0.2, color=WIND_COLOR, label='P10–P90 band')
ax.fill_between(hourly['hour'], hourly['p25'], hourly['p75'], alpha=0.35, color=WIND_COLOR, label='P25–P75 band')
ax.plot(hourly['hour'], hourly['median'], color=WIND_COLOR, linewidth=2.5, label='Median')
ax.plot(hourly['hour'], hourly['mean'], color=ACTUAL_COLOR, linewidth=2, linestyle='--', label='Mean')
ax.plot(hourly['hour'], hourly['p10'], color=NEG_COLOR, linewidth=1.5, linestyle=':', label='P10 (firm)')

ax.set_xlabel('Hour of Day (UTC)')
ax.set_ylabel('Generation (MW)')
ax.set_title('Wind Generation by Hour of Day — Distribution Bands')
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 24, 2)])
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('generation_by_hour.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: generation_by_hour.png')

## 6. Low-Wind Event Analysis

Wind droughts (prolonged periods of low generation) are a key risk for grid operators.

In [ ]:
# Find consecutive periods below a threshold
THRESHOLD = gen.quantile(0.10)  # P10 as 'low wind' threshold
print(f'Low-wind threshold (P10): {THRESHOLD:.0f} MW')

df_sorted = df.sort_values('startTime').copy()
df_sorted['low_wind'] = df_sorted['generation'] < THRESHOLD

# Identify consecutive low-wind spells
df_sorted['spell_id'] = (df_sorted['low_wind'] != df_sorted['low_wind'].shift()).cumsum()
low_spells = df_sorted[df_sorted['low_wind']].groupby('spell_id').agg(
    start=('startTime', 'min'),
    end=('startTime', 'max'),
    n=('generation', 'count'),
    min_gen=('generation', 'min'),
    avg_gen=('generation', 'mean'),
).reset_index(drop=True)
low_spells['duration_h'] = low_spells['n'] * 0.5

print(f'\nLow-wind spells (below {THRESHOLD:.0f} MW):')
print(f'  Total spells: {len(low_spells)}')
print(f'  Longest spell: {low_spells["duration_h"].max():.1f} hours')
print(f'  Average spell: {low_spells["duration_h"].mean():.1f} hours')
print(f'  Total low-wind hours: {low_spells["duration_h"].sum():.0f} h of {len(df_sorted)*0.5:.0f} h total')
print()
print('Top 5 longest low-wind spells:')
print(low_spells.nlargest(5, 'duration_h')[['start', 'end', 'duration_h', 'min_gen', 'avg_gen']].to_string(index=False))

## 7. Recommendation: Reliable Wind Capacity

**Framework**: We use the **firm capacity** concept — the level of generation that can be relied upon with a defined probability.

In [ ]:
p5  = gen.quantile(0.05)
p10 = gen.quantile(0.10)
p50 = gen.quantile(0.50)
mean_gen = gen.mean()
max_gen = gen.max()

# Capacity factor estimate
# UK installed wind capacity as of early 2024: ~30,000 MW (onshore + offshore)
INSTALLED_CAPACITY_MW = 30000
capacity_factor = mean_gen / INSTALLED_CAPACITY_MW

fig, ax = plt.subplots(figsize=(10, 6))

levels = {
    'Maximum (Jan 2024)': (max_gen, '#8dab82', ':'),
    'Mean': (mean_gen, ACTUAL_COLOR, '-'),
    'Median (P50)': (p50, WARN_COLOR, '--'),
    'Firm capacity (P10)': (p10, WIND_COLOR, '--'),
    'Conservative firm (P5)': (p5, NEG_COLOR, '--'),
}

sorted_gen2 = np.sort(gen)
cdf = np.arange(1, len(sorted_gen2)+1) / len(sorted_gen2) * 100
ax.plot(sorted_gen2, cdf, color='#e8f5e2', linewidth=2.5, label='CDF (% of time ≤ x MW)')

for label, (val, color, ls) in levels.items():
    ax.axvline(val, color=color, linestyle=ls, linewidth=2, label=f'{label}: {val:.0f} MW')

ax.set_xlabel('Wind Generation (MW)')
ax.set_ylabel('% of half-hours')
ax.set_title('Recommended Firm Capacity Levels — UK Wind (Jan 2024)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(loc='upper left', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('firm_capacity.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print('Saved: firm_capacity.png')

In [ ]:
print('=' * 65)
print('WIND RELIABILITY RECOMMENDATION — UK NATIONAL WIND (Jan 2024)')
print('=' * 65)
print()
print('OBSERVED GENERATION STATISTICS:')
print(f'  Mean generation:     {mean_gen:,.0f} MW')
print(f'  Median (P50):        {p50:,.0f} MW  — exceeded 50% of the time')
print(f'  P10 (firm):          {p10:,.0f} MW  — exceeded 90% of the time')
print(f'  P5 (conservative):   {p5:,.0f} MW  — exceeded 95% of the time')
print(f'  Maximum:             {max_gen:,.0f} MW')
print(f'  Capacity factor:     {capacity_factor:.1%}  (assuming {INSTALLED_CAPACITY_MW:,} MW installed)')
print()
print('RECOMMENDATION:')
print(f'  ► For reliable grid planning, commit {p10:,.0f} MW (P10) as the')
print(f'    firm wind contribution to electricity demand.')
print(f'    This is available 90% of half-hour periods in January 2024.')
print()
print('REASONING:')
print(f'  1. The P10 value ({p10:,.0f} MW) represents a conservative but')
print(f'     practical floor — the grid operator can count on at least')
print(f'     this much generation in 9 out of 10 half-hour intervals.')
print()
print(f'  2. For critical planning (e.g., capacity auctions), the P5')
print(f'     value of {p5:,.0f} MW provides a 95% confidence level.')
print()
print(f'  3. January is a high-wind month in the UK. Applying the same')
print(f'     methodology to a full year would likely yield lower P10')
print(f'     values (~1,500–3,000 MW) due to summer low-wind periods.')
print()
print(f'  4. Low-wind spells (below P10) last on average')
print(f'     {low_spells["duration_h"].mean():.1f} hours, peaking at {low_spells["duration_h"].max():.0f} hours.')
print(f'     This informs how much backup capacity or storage is needed.')
print()
print(f'  5. The mean capacity factor of {capacity_factor:.1%} is typical for')
print(f'     UK wind in winter — higher than the annual average (~30%).')
print('=' * 65)